# FRetinalNet — Interactive Demo

Run retinal vessel segmentation inference on bundled DRIVE test samples with the pretrained **FRetinalNet**.

- Runtime: `Runtime > Change runtime type > GPU (T4)` recommended (works on CPU, just slower).
- The demo uses **3 bundled DRIVE test images** (`assets/samples/`) so it runs out of the box.
  For the full 20-image test protocol, see the [repository README](https://github.com/SuperXionghaoren/FRetinalNet).
- Sample images belong to the [DRIVE dataset](https://drive.grand-challenge.org/) owners and are bundled solely for demonstration.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 1) get the repository (skipped if the notebook already runs inside it)
import os, subprocess

REPO_URL = 'https://github.com/SuperXionghaoren/FRetinalNet.git'
if not os.path.isdir('fretinalnet_notebook'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    os.chdir('FRetinalNet')
print('working directory:', os.getcwd())

In [ ]:
# 2) install the few dependencies (Colab already ships most of them)
!pip install -q opencv-python scikit-image pandas

In [ ]:
# 3) download the pretrained weights from Hugging Face (~2.9 GB, one-time)
# In mainland China: uncomment the next line to route through the HF mirror
# import os as _os; _os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
WEIGHTS_NAME = 'dice_fdconv_drive_ori.pth'
os.makedirs('weights', exist_ok=True)
weights_path = os.path.join('weights', WEIGHTS_NAME)
if not os.path.exists(weights_path):
    from huggingface_hub import hf_hub_download
    HF_REPO_ID = os.environ.get('HF_REPO_ID', 'waspwallbvb/FRetinalNet')
    weights_path = hf_hub_download(repo_id=HF_REPO_ID, filename=WEIGHTS_NAME, local_dir='weights')
print('weights:', weights_path)

## Bundled samples

Three DRIVE **test** images (`02 / 06 / 15`) with their `1st_manual` annotations are bundled in
`assets/samples/` so this demo runs without any dataset download. The preprocessing and
threshold (512x512 + CLAHE, binarization at 0.5) are **identical** to the full evaluation protocol.

In [ ]:
# 4) build the demo dataset from the bundled samples
from data.drive import DRIVE_Test_Dataset, transform

dataset = DRIVE_Test_Dataset('assets/samples/images', 'assets/samples/1st_manual', transform=transform)
print(f'{len(dataset)} sample images')

In [ ]:
# 5) load the model (paper naming: FRetinalNet with FFTBlock / FaFBlock / ResNetEncoder)
import torch
from fretinalnet_notebook import FRetinalNet

model = FRetinalNet(num_classes=1)
state = torch.load(weights_path, map_location='cpu')
model.load_state_dict(state, strict=True)
model.to(device).eval()
print('model loaded (strict=True ok)')

In [ ]:
# 6) inference + metrics (same metric implementation as eval_test.py)
import numpy as np
import pandas as pd
import torch
from eval_test import compute_batch_metrics, KEYS

rows = []
probs = {}
with torch.no_grad():
    for idx in range(len(dataset)):
        name = os.path.basename(dataset.images[idx]).replace('_test.tif', '')
        image, mask = dataset[idx]
        image, mask = image.unsqueeze(0).to(device), mask.unsqueeze(0).to(device)
        out1, out2 = model(image)          # forward returns (fused_out, fused_out_low)
        prob = torch.sigmoid(out1)         # main (image-branch) prediction head
        probs[name] = (prob[0, 0].cpu().numpy(), mask[0, 0].cpu().numpy(),
                       image[0].cpu().numpy().transpose(1, 2, 0))
        rows.append({'image': name, **compute_batch_metrics(prob, mask)})

df = pd.DataFrame(rows)[['image'] + KEYS].round(4)
print(df.to_string(index=False))
print('\nmean  ' + '  '.join(f'{k}={df[k].mean():.4f}' for k in KEYS))

In [ ]:
# 7) visualization: input | probability heatmap | overlay | GT | prediction
import matplotlib.pyplot as plt
import numpy as np

threshold = 0.5
fig, axes = plt.subplots(len(probs), 5, figsize=(20, 4 * len(probs)))
if len(probs) == 1:
    axes = axes[None, :]
for row, (name, (prob, gt, img)) in enumerate(probs.items()):
    img8 = (img * 255).clip(0, 255).astype(np.uint8)
    red = img8.copy(); red[..., 0] = np.clip(0.55 * red[..., 0] + prob * 255, 0, 255)
    overlay = (0.55 * img8 + 0.45 * red).astype(np.uint8)

    axes[row][0].imshow(img8);                        axes[row][0].set_title(f'{name}: input')
    axes[row][1].imshow(prob, cmap='gray');           axes[row][1].set_title('probability')
    axes[row][2].imshow(overlay);                     axes[row][2].set_title('overlay')
    axes[row][3].imshow(gt, cmap='gray');             axes[row][3].set_title('ground truth')
    axes[row][4].imshow((prob >= threshold).astype(float), cmap='gray')
    axes[row][4].set_title('prediction (0.5)')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout(); plt.savefig('demo_result.png', dpi=120, bbox_inches='tight'); plt.show()
print('figure saved to demo_result.png')

## Next steps

- **Full protocol** (all 20 DRIVE test images): register at
  [drive.grand-challenge.org](https://drive.grand-challenge.org/), put the dataset under
  `data/DRIVE` (see `scripts/prepare_drive.py`), then run `python eval_test.py`.
  Expected: Dice **0.8648** / IoU 0.7669 / ACC 0.9703 (best single checkpoint;
  the paper's Table 2 reports the 3-run average 0.8578).
